# Fase 1 - Densidad de datos de accesibilidad en Madrid

Este notebook analiza la cobertura de OpenStreetMap y Mapillary en el interior de la M-30 para decidir una zona piloto del MVP de rutas accesibles.

Objetivos:

- Medir disponibilidad de atributos de accesibilidad por zona.
- Separar datos estructurados OSM de cobertura visual Mapillary.
- Identificar atributos disponibles, ausentes e inciertos.
- Generar mapas de densidad y CSVs reproducibles.
- Proponer zonas candidatas para el area piloto.

Notas de seguridad del TFM:

- La ausencia de datos no implica ausencia de barreras.
- Este analisis mide cobertura de datos, no accesibilidad real.
- Los atributos desconocidos deben reducir confianza o generar aviso en el MVP.


## 0. Preparacion

Requisitos recomendados desde la raiz del repo:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m ipykernel install --user --name tfm-density --display-name "TFM density"
```

Mapillary requiere token. Crear un `.env` local, no versionado:

```text
MAPILLARY_ACCESS_TOKEN=...
```

Si no hay token, el notebook ejecuta la parte OSM y deja Mapillary marcado como no disponible. Por defecto Mapillary descarga hasta 5 paginas para evitar llamadas masivas; se puede ajustar con `MAPILLARY_MAX_PAGES` en `.env`.


In [27]:
from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import requests
from branca.colormap import LinearColormap
from dotenv import load_dotenv
from folium.features import GeoJsonTooltip
from folium.plugins import GroupedLayerControl
from shapely.geometry import Point, box
from tqdm.auto import tqdm

load_dotenv()

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
MAPS_DIR = REPO_ROOT / "outputs" / "maps"
for directory in (DATA_RAW, DATA_PROCESSED, MAPS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

WGS84 = "EPSG:4326"
MADRID_CRS = "EPSG:25830"  # ETRS89 / UTM zone 30N, suitable for Madrid distances.

MADRID_CENTER = (40.4168, -3.7038)
GRID_SIZE_METERS = 1000

M30_BOUNDARY_URL = (
    "https://services5.arcgis.com/CQghJ1IcnV6WlQrf/ArcGIS/rest/services/"
    "ZBE_Madrid_M30/FeatureServer/32/query"
)
OVERPASS_URLS = (
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
)
MAPILLARY_TOKEN = os.getenv("MAPILLARY_ACCESS_TOKEN")
MAPILLARY_MAX_PAGES = int(os.getenv("MAPILLARY_MAX_PAGES", "5"))

pd.set_option("display.max_columns", 80)


## 1. Atributos de accesibilidad a medir

Los atributos se eligen porque son relevantes para rutas de personas ciegas o con baja vision y porque existen como senales observables o tags OSM verificables. No se inventan tags: si un atributo no aparece en OSM, se marca como ausente/desconocido.


In [28]:
@dataclass(frozen=True)
class OsmAttributeQuery:
    name: str
    description: str
    overpass_selector: str
    safety_note: str


OSM_ATTRIBUTE_QUERIES = [
    OsmAttributeQuery(
        "crossings",
        "Pasos de peatones etiquetados como highway=crossing.",
        'nwr["highway"="crossing"]',
        "La presencia de cruce no confirma que sea seguro ni accesible.",
    ),
    OsmAttributeQuery(
        "traffic_signals",
        "Semaforos etiquetados como highway=traffic_signals.",
        'nwr["highway"="traffic_signals"]',
        "No todos los semaforos tienen senal acustica.",
    ),
    OsmAttributeQuery(
        "audible_signals",
        "Semaforos o cruces con senal acustica/vibracion declarada.",
        'nwr["traffic_signals:sound"];nwr["traffic_signals:vibration"]',
        "Atributo critico: la ausencia de tag no implica ausencia real.",
    ),
    OsmAttributeQuery(
        "tactile_paving",
        "Pavimento tactil declarado.",
        'nwr["tactile_paving"]',
        "Atributo muy incompleto en muchas ciudades.",
    ),
    OsmAttributeQuery(
        "kerbs",
        "Bordillos, rebajes o kerbs declarados.",
        'nwr["kerb"]',
        "La semantica del valor debe analizarse antes de puntuar accesibilidad.",
    ),
    OsmAttributeQuery(
        "sidewalks",
        "Calles o vias con atributo sidewalk.",
        'way["sidewalk"]',
        "Sidewalk=yes/left/right/no requiere interpretacion por tramo.",
    ),
    OsmAttributeQuery(
        "ramps_or_wheelchair",
        "Rampas o accesibilidad wheelchair declarada.",
        'nwr["ramp"];nwr["wheelchair"]',
        "No debe asumirse que wheelchair=yes equivale a ruta segura para todos.",
    ),
    OsmAttributeQuery(
        "steps",
        "Escaleras etiquetadas como highway=steps.",
        'nwr["highway"="steps"]',
        "Puede ser barrera critica segun perfil del usuario.",
    ),
    OsmAttributeQuery(
        "surface_on_pedestrian_ways",
        "Superficie declarada en vias peatonales o calles relevantes.",
        'way["highway"~"^(footway|path|pedestrian|steps|living_street|residential|service|unclassified|tertiary|secondary|primary)$"]["surface"]',
        "La superficie puede afectar seguridad, pero los valores deben normalizarse.",
    ),
    OsmAttributeQuery(
        "incline",
        "Pendiente declarada mediante incline.",
        'nwr["incline"]',
        "Dato escaso; para produccion convendria combinar con DEM.",
    ),
]

pd.DataFrame([query.__dict__ for query in OSM_ATTRIBUTE_QUERIES])


,name,description,overpass_selector,safety_note
0,crossings,Pasos de peatones etiquetados como highway=cro...,"nwr[""highway""=""crossing""]",La presencia de cruce no confirma que sea segu...
1,traffic_signals,Semaforos etiquetados como highway=traffic_sig...,"nwr[""highway""=""traffic_signals""]",No todos los semaforos tienen senal acustica.
2,audible_signals,Semaforos o cruces con senal acustica/vibracio...,"nwr[""traffic_signals:sound""];nwr[""traffic_sign...",Atributo critico: la ausencia de tag no implic...
3,tactile_paving,Pavimento tactil declarado.,"nwr[""tactile_paving""]",Atributo muy incompleto en muchas ciudades.
4,kerbs,"Bordillos, rebajes o kerbs declarados.","nwr[""kerb""]",La semantica del valor debe analizarse antes d...
5,sidewalks,Calles o vias con atributo sidewalk.,"way[""sidewalk""]",Sidewalk=yes/left/right/no requiere interpreta...
6,ramps_or_wheelchair,Rampas o accesibilidad wheelchair declarada.,"nwr[""ramp""];nwr[""wheelchair""]",No debe asumirse que wheelchair=yes equivale a...
7,steps,Escaleras etiquetadas como highway=steps.,"nwr[""highway""=""steps""]",Puede ser barrera critica segun perfil del usu...
8,surface_on_pedestrian_ways,Superficie declarada en vias peatonales o call...,"way[""highway""~""^(footway|path|pedestrian|steps...","La superficie puede afectar seguridad, pero lo..."
9,incline,Pendiente declarada mediante incline.,"nwr[""incline""]",Dato escaso; para produccion convendria combin...


## 2. Ambito y malla de analisis

El ambito se limita al poligono interior de la M-30 publicado en la capa `M30_FeatureToPolygon`. Se usa una malla regular de 1 km recortada por ese limite; las celdas de borde conservan solo su superficie interior, para que las densidades no incluyan zonas exteriores.


In [29]:
def load_m30_boundary() -> gpd.GeoDataFrame:
    """Load and cache the polygon representing the area inside the M-30.

    Returns:
        Single-row GeoDataFrame containing the M-30 interior boundary.
    """
    cache_path = DATA_RAW / "m30_interior_boundary.geojson"
    if cache_path.exists():
        boundary = gpd.read_file(cache_path)
    else:
        params = {
            "where": "1=1",
            "outFields": "OBJECTID",
            "returnGeometry": "true",
            "outSR": "4326",
            "f": "geojson",
        }
        response = requests.get(M30_BOUNDARY_URL, params=params, timeout=120)
        response.raise_for_status()
        payload = response.json()
        if not payload.get("features"):
            raise ValueError("The M-30 boundary service returned no features.")
        cache_path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
        boundary = gpd.GeoDataFrame.from_features(payload["features"], crs=WGS84)

    boundary = boundary.to_crs(WGS84)
    geometry = boundary.geometry.make_valid().union_all()
    if geometry.is_empty:
        raise ValueError("The M-30 boundary geometry is empty.")
    return gpd.GeoDataFrame({"area_name": ["Interior M-30"]}, geometry=[geometry], crs=WGS84)


def build_grid(boundary: gpd.GeoDataFrame, cell_size_m: int) -> gpd.GeoDataFrame:
    """Build a regular grid clipped to a study-area boundary.

    Args:
        boundary: Study-area polygon in WGS84.
        cell_size_m: Cell size in meters in the projected CRS.

    Returns:
        GeoDataFrame with one polygon per analysis zone.
    """
    projected_boundary = boundary.to_crs(MADRID_CRS)
    minx, miny, maxx, maxy = projected_boundary.total_bounds

    cells = []
    y = miny
    row = 0
    while y < maxy:
        x = minx
        col = 0
        while x < maxx:
            cells.append({"zone_id": f"z{row:02d}_{col:02d}", "geometry": box(x, y, x + cell_size_m, y + cell_size_m)})
            x += cell_size_m
            col += 1
        y += cell_size_m
        row += 1

    grid = gpd.GeoDataFrame(cells, crs=MADRID_CRS)
    grid = gpd.clip(grid, projected_boundary)
    grid = grid.loc[~grid.geometry.is_empty].copy()
    grid["area_km2"] = grid.geometry.area / 1_000_000
    return grid.to_crs(WGS84)


m30_boundary = load_m30_boundary()
min_lon, min_lat, max_lon, max_lat = m30_boundary.total_bounds
MADRID_BBOX = (min_lat, min_lon, max_lat, max_lon)
grid = build_grid(m30_boundary, GRID_SIZE_METERS)
grid.head()


,zone_id,geometry,area_km2
10,z01_02,"POLYGON ((-3.71455 40.40671, -3.71448 40.40022...",0.436085
18,z02_02,"POLYGON ((-3.71464 40.41572, -3.71455 40.40671...",0.633323
26,z03_02,"POLYGON ((-3.72653 40.42465, -3.71474 40.42473...",0.716237
6,z00_06,"POLYGON ((-3.6791 40.39791, -3.67007 40.39797,...",0.312302
5,z00_05,"POLYGON ((-3.69089 40.39784, -3.6791 40.39791,...",0.939383


In [30]:
grid.explore(
    tiles="CartoDB positron",
    tooltip=["zone_id", "area_km2"],
    style_kwds={"fillOpacity": 0.05, "weight": 0.5},
)


## 3. Descarga reproducible de OpenStreetMap via Overpass

Cada consulta se guarda en `data/raw/osm_<atributo>.json` para poder reanalizar sin repetir llamadas. Se usa `out center tags` para estimar densidad espacial con puntos representativos; este notebook no pretende reconstruir geometrias completas de routing.


In [31]:
def _post_overpass_query(query: str) -> dict[str, Any]:
    """Send one query, trying the configured Overpass endpoints in order."""
    headers = {
        "User-Agent": "TFM-Accessible-Madrid/0.1 (academic research)",
        "Accept": "application/json",
    }
    retryable_statuses = {429, 502, 503, 504}
    errors = []
    for endpoint in OVERPASS_URLS:
        try:
            response = requests.post(
                endpoint, data={"data": query}, headers=headers, timeout=240
            )
        except requests.RequestException as exc:
            errors.append(f"{endpoint}: {exc}")
            continue

        if response.ok:
            return response.json()

        error_excerpt = response.text[:300].strip().replace("\n", " ")
        errors.append(f"{endpoint}: HTTP {response.status_code} {error_excerpt}")
        if response.status_code not in retryable_statuses:
            break
        time.sleep(2)

    raise requests.HTTPError("All Overpass endpoints failed: " + " | ".join(errors))


def _split_bbox(
    bbox: tuple[float, float, float, float], divisions: int
) -> list[tuple[float, float, float, float]]:
    """Split a south-west-north-east bounding box into equal tiles."""
    south, west, north, east = bbox
    latitude_step = (north - south) / divisions
    longitude_step = (east - west) / divisions
    return [
        (
            south + row * latitude_step,
            west + column * longitude_step,
            south + (row + 1) * latitude_step,
            west + (column + 1) * longitude_step,
        )
        for row in range(divisions)
        for column in range(divisions)
    ]


def _build_overpass_query(
    attribute: OsmAttributeQuery, bbox: tuple[float, float, float, float]
) -> str:
    """Build an Overpass QL query for one attribute and bounding box."""
    south, west, north, east = bbox
    selector_lines = []
    for selector in attribute.overpass_selector.split(";"):
        selector = selector.strip()
        if selector:
            selector_lines.append(f"  {selector}({south},{west},{north},{east});")
    return "\n".join([
        "[out:json][timeout:180];",
        "(",
        *selector_lines,
        ");",
        "out center tags;",
    ])


def _merge_overpass_payloads(payloads: list[dict[str, Any]]) -> dict[str, Any]:
    """Merge Overpass responses and remove duplicate OSM elements."""
    merged = dict(payloads[-1]) if payloads else {"elements": []}
    elements_by_id = {}
    for payload in payloads:
        for element in payload.get("elements", []):
            elements_by_id[(element.get("type"), element.get("id"))] = element
    merged["elements"] = list(elements_by_id.values())
    return merged


def _fetch_overpass_tile(
    attribute: OsmAttributeQuery,
    bbox: tuple[float, float, float, float],
    cache_label: str,
    depth: int = 0,
) -> dict[str, Any]:
    """Fetch one tile, subdividing and caching it if public servers fail."""
    tile_cache_path = DATA_RAW / f"osm_{attribute.name}_tile_{cache_label}.json"
    if tile_cache_path.exists():
        return json.loads(tile_cache_path.read_text(encoding="utf-8"))

    try:
        payload = _post_overpass_query(_build_overpass_query(attribute, bbox))
    except requests.HTTPError:
        if depth >= 2:
            raise
        tqdm.write(f"{attribute.name}: subdividing tile {cache_label}")
        time.sleep(10)
        child_payloads = [
            _fetch_overpass_tile(
                attribute, child_bbox, f"{cache_label}_{child_number:02d}", depth + 1
            )
            for child_number, child_bbox in enumerate(_split_bbox(bbox, 2), start=1)
        ]
        payload = _merge_overpass_payloads(child_payloads)

    tile_cache_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    time.sleep(5)  # Reduce pressure and rate-limit risk on public instances.
    return payload


def overpass_query(attribute: OsmAttributeQuery, bbox: tuple[float, float, float, float]) -> dict[str, Any]:
    """Run or load an Overpass query for one accessibility attribute.

    Args:
        attribute: Attribute query definition.
        bbox: Tuple with south, west, north, east coordinates.

    Returns:
        Raw Overpass JSON response.
    """
    cache_path = DATA_RAW / f"osm_{attribute.name}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    divisions = 3 if attribute.name == "surface_on_pedestrian_ways" else 1
    tiles = _split_bbox(bbox, divisions)
    elements_by_id = {}
    payload = {}
    tile_iterator = (
        tqdm(tiles, desc=f"{attribute.name} tiles", leave=False)
        if divisions > 1
        else tiles
    )

    for tile_number, tile_bbox in enumerate(tile_iterator, start=1):
        if divisions > 1:
            tile_payload = _fetch_overpass_tile(
                attribute, tile_bbox, f"{tile_number:02d}"
            )
        else:
            tile_payload = _post_overpass_query(
                _build_overpass_query(attribute, tile_bbox)
            )
            time.sleep(2)  # Be gentle with Overpass.

        payload = tile_payload
        for element in tile_payload.get("elements", []):
            elements_by_id[(element.get("type"), element.get("id"))] = element

    payload["elements"] = list(elements_by_id.values())
    cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return payload


def osm_elements_to_points(payload: dict[str, Any], attribute_name: str) -> gpd.GeoDataFrame:
    """Convert Overpass elements to representative points.

    Args:
        payload: Raw Overpass JSON response.
        attribute_name: Name assigned to all returned elements.

    Returns:
        GeoDataFrame with representative points and OSM tags.
    """
    rows = []
    for element in payload.get("elements", []):
        lon = element.get("lon")
        lat = element.get("lat")
        if lon is None or lat is None:
            center = element.get("center") or {}
            lon = center.get("lon")
            lat = center.get("lat")
        if lon is None or lat is None:
            continue
        rows.append({
            "source": "osm",
            "attribute": attribute_name,
            "osm_type": element.get("type"),
            "osm_id": element.get("id"),
            "tags": element.get("tags", {}),
            "geometry": Point(lon, lat),
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=WGS84)


In [32]:
osm_layers = []
for attribute in tqdm(OSM_ATTRIBUTE_QUERIES, desc="OSM attributes"):
    payload = overpass_query(attribute, MADRID_BBOX)
    layer = osm_elements_to_points(payload, attribute.name)
    print(f"{attribute.name}: {len(layer):,} elements")
    osm_layers.append(layer)

osm_points = pd.concat(osm_layers, ignore_index=True) if osm_layers else gpd.GeoDataFrame(geometry=[], crs=WGS84)
osm_points = gpd.GeoDataFrame(osm_points, geometry="geometry", crs=WGS84)
osm_points = gpd.sjoin(
    osm_points, m30_boundary[["geometry"]], how="inner", predicate="within"
).drop(columns="index_right")
osm_points["tags_json"] = osm_points["tags"].apply(lambda tags: json.dumps(tags, ensure_ascii=False, sort_keys=True))
osm_points.drop(columns=["tags"]).to_parquet(DATA_PROCESSED / "osm_accessibility_points.parquet", index=False)
osm_points.head()


OSM attributes:  40%|████      | 4/10 [00:00<00:00, 34.63it/s]

crossings: 12,187 elements
traffic_signals: 3,950 elements
audible_signals: 2,441 elements
tactile_paving: 12,117 elements
kerbs: 4,373 elements
sidewalks: 4,351 elements


OSM attributes: 100%|██████████| 10/10 [00:00<00:00, 22.10it/s]

ramps_or_wheelchair: 9,604 elements
steps: 2,740 elements
surface_on_pedestrian_ways: 31,236 elements
incline: 1,695 elements


,source,attribute,osm_type,osm_id,tags,geometry,tags_json
0,osm,crossings,node,20953235,"{'button_operated': 'no', 'crossing': 'traffic...",POINT (-3.69399 40.41916),"{""button_operated"": ""no"", ""crossing"": ""traffic..."
1,osm,crossings,node,21473037,"{'button_operated': 'yes', 'crossing': 'traffi...",POINT (-3.72723 40.42637),"{""button_operated"": ""yes"", ""crossing"": ""traffi..."
4,osm,crossings,node,21734250,"{'check_date:crossing': '2023-05-30', 'crossin...",POINT (-3.70802 40.41612),"{""check_date:crossing"": ""2023-05-30"", ""crossin..."
5,osm,crossings,node,21734252,"{'crossing': 'uncontrolled', 'crossing:marking...",POINT (-3.70847 40.41677),"{""crossing"": ""uncontrolled"", ""crossing:marking..."
6,osm,crossings,node,21777429,"{'button_operated': 'no', 'crossing': 'traffic...",POINT (-3.72068 40.40404),"{""button_operated"": ""no"", ""crossing"": ""traffic..."


## 4. Cobertura visual de Mapillary

Mapillary se usa aqui como proxy de cobertura visual disponible, no como fuente de verdad de accesibilidad. La deteccion automatica de atributos mediante VLM queda para una fase final.


In [33]:
def _strip_access_token(url: str | None) -> str | None:
    """Remove access-token parameters from a Mapillary pagination URL."""
    if not url:
        return None
    parts = urlsplit(url)
    safe_query = urlencode(
        [(key, value) for key, value in parse_qsl(parts.query) if key != "access_token"]
    )
    return urlunsplit((parts.scheme, parts.netloc, parts.path, safe_query, parts.fragment))


def _get_mapillary_page(
    url: str, params: dict[str, Any] | None, token: str
) -> dict[str, Any]:
    """Fetch one Mapillary page without exposing the token in the request URL."""
    headers = {
        "Authorization": f"OAuth {token}",
        "User-Agent": "TFM-Accessible-Madrid/0.1 (academic research)",
        "Accept": "application/json",
    }
    retryable_statuses = {429, 500, 502, 503, 504}
    last_error = "unknown error"
    for attempt in range(3):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=120)
        except requests.RequestException as exc:
            last_error = type(exc).__name__
        else:
            if response.ok:
                return response.json()
            try:
                error_message = response.json().get("error", {}).get("message")
            except ValueError:
                error_message = None
            last_error = f"HTTP {response.status_code}: {error_message or 'no details'}"
            if response.status_code not in retryable_statuses:
                break
        time.sleep(2 ** (attempt + 1))

    raise requests.HTTPError(
        f"Mapillary request failed after 3 attempts: {last_error}. Token omitted."
    )


def _fetch_mapillary_tile(
    bbox: tuple[float, float, float, float],
    token: str,
    limit: int,
    max_pages: int,
    cache_label: str,
    depth: int = 0,
) -> list[dict[str, Any]]:
    """Fetch and cache one tile, subdividing dense areas adaptively."""
    tile_cache_path = DATA_RAW / f"mapillary_images_m30_tile_{cache_label}.json"
    if tile_cache_path.exists():
        checkpoint = json.loads(tile_cache_path.read_text(encoding="utf-8"))
    else:
        checkpoint = {"data": [], "next": None, "pages": 0, "complete": False}

    if checkpoint.get("complete"):
        return checkpoint.get("data", [])

    south, west, north, east = bbox
    tile_data = checkpoint.get("data", [])
    page_number = int(checkpoint.get("pages", 0))
    url = checkpoint.get("next") or "https://graph.mapillary.com/images"
    params = None if checkpoint.get("next") else {
        "bbox": f"{west},{south},{east},{north}",
        "fields": "id,geometry,captured_at",
        "limit": limit,
    }

    try:
        while page_number < max_pages:
            page = _get_mapillary_page(url, params, token)
            tile_data.extend(page.get("data", []))
            next_url = _strip_access_token(page.get("paging", {}).get("next"))
            page_number += 1
            complete = not next_url or page_number >= max_pages
            checkpoint = {
                "data": tile_data,
                "next": next_url,
                "pages": page_number,
                "complete": complete,
            }
            tile_cache_path.write_text(
                json.dumps(checkpoint, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            if complete:
                return tile_data
            url = next_url or url
            params = None
            time.sleep(1)
    except requests.HTTPError as exc:
        can_subdivide = "reduce the amount of data" in str(exc) or "HTTP 500" in str(exc)
        if depth >= 2 or not can_subdivide:
            raise
        tqdm.write(f"Mapillary: subdividing tile {cache_label}")
        child_data = []
        for child_number, child_bbox in enumerate(_split_bbox(bbox, 2), start=1):
            child_data.extend(
                _fetch_mapillary_tile(
                    child_bbox,
                    token,
                    limit,
                    max_pages,
                    f"{cache_label}_{child_number:02d}",
                    depth + 1,
                )
            )
        unique_data = {item.get("id"): item for item in child_data}
        tile_data = list(unique_data.values())
        checkpoint = {
            "data": tile_data,
            "next": None,
            "pages": 0,
            "complete": True,
            "subdivided": True,
        }
        tile_cache_path.write_text(
            json.dumps(checkpoint, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        return tile_data

    return tile_data


def fetch_mapillary_images(bbox: tuple[float, float, float, float], token: str | None, limit: int = 100, max_pages: int = 5) -> gpd.GeoDataFrame:
    """Fetch Mapillary image points inside the Madrid bounding box.

    Args:
        bbox: Tuple with south, west, north, east coordinates.
        token: Mapillary access token. If missing, returns an empty frame.
        limit: Page size requested from the API for each tile.
        max_pages: Maximum pages per tile for reproducibility and API hygiene.

    Returns:
        GeoDataFrame with Mapillary image points.
    """
    cache_path = DATA_RAW / "mapillary_images_m30.json"
    if cache_path.exists():
        payload = json.loads(cache_path.read_text(encoding="utf-8"))
    else:
        if not token:
            print("MAPILLARY_ACCESS_TOKEN is missing. Mapillary coverage will be empty.")
            return gpd.GeoDataFrame(columns=["source", "image_id", "captured_at", "geometry"], geometry="geometry", crs=WGS84)

        images_by_id = {}
        tiles = _split_bbox(bbox, 4)
        for tile_number, tile_bbox in enumerate(
            tqdm(tiles, desc="Mapillary tiles"), start=1
        ):
            tile_data = _fetch_mapillary_tile(
                tile_bbox, token, limit, max_pages, f"{tile_number:02d}"
            )

            for item in tile_data:
                images_by_id[item.get("id")] = item

        payload = {"data": list(images_by_id.values())}
        cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    rows = []
    for item in payload.get("data", []):
        coordinates = (item.get("geometry") or {}).get("coordinates")
        if not coordinates:
            continue
        lon, lat = coordinates
        rows.append({
            "source": "mapillary",
            "image_id": item.get("id"),
            "captured_at": item.get("captured_at"),
            "geometry": Point(lon, lat),
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=WGS84)


mapillary_points = fetch_mapillary_images(MADRID_BBOX, MAPILLARY_TOKEN, max_pages=MAPILLARY_MAX_PAGES)
if not mapillary_points.empty:
    mapillary_points = gpd.sjoin(
        mapillary_points, m30_boundary[["geometry"]], how="inner", predicate="within"
    ).drop(columns="index_right")
mapillary_points.to_parquet(DATA_PROCESSED / "mapillary_image_points.parquet", index=False)
mapillary_points.head()


,source,image_id,captured_at,geometry
85,mapillary,1173414123676941,1715176644733,POINT (-3.70869 40.41078)
86,mapillary,448542154336429,1715176635833,POINT (-3.70861 40.4107)
87,mapillary,454860830452621,1715176639781,POINT (-3.70863 40.41075)
88,mapillary,809283827334379,1715176654700,POINT (-3.70879 40.41083)
89,mapillary,1181897783002409,1715176650729,POINT (-3.70876 40.41082)


## 5. Métricas de cobertura por zona

Se calculan conteos y densidades por km2 para cada atributo OSM y para imagenes Mapillary. También se calcula riqueza de atributos: numero de atributos distintos presentes en cada zona.


In [34]:
def count_points_by_zone(points: gpd.GeoDataFrame, zones: gpd.GeoDataFrame, value_column: str, prefix: str) -> pd.DataFrame:
    """Count point records per analysis zone.

    Args:
        points: Point GeoDataFrame in WGS84.
        zones: Polygon GeoDataFrame in WGS84.
        value_column: Column to split counts by.
        prefix: Prefix for generated count columns.

    Returns:
        DataFrame keyed by zone_id with count columns.
    """
    if points.empty:
        return pd.DataFrame({"zone_id": zones["zone_id"]})

    joined = gpd.sjoin(points, zones[["zone_id", "geometry"]], how="inner", predicate="within")
    counts = joined.groupby(["zone_id", value_column]).size().unstack(fill_value=0)
    counts = counts.add_prefix(prefix)
    return counts.reset_index()


zone_metrics = grid[["zone_id", "area_km2", "geometry"]].copy()

osm_counts = count_points_by_zone(osm_points, grid, "attribute", "osm_")
zone_metrics = zone_metrics.merge(osm_counts, on="zone_id", how="left")

if not mapillary_points.empty:
    mapillary_with_kind = mapillary_points.assign(kind="images")
    mapillary_counts = count_points_by_zone(mapillary_with_kind, grid, "kind", "mapillary_")
    zone_metrics = zone_metrics.merge(mapillary_counts, on="zone_id", how="left")
else:
    zone_metrics["mapillary_images"] = 0

count_columns = [column for column in zone_metrics.columns if column.startswith("osm_") or column.startswith("mapillary_")]
zone_metrics[count_columns] = zone_metrics[count_columns].fillna(0).astype(int)

osm_count_columns = [column for column in zone_metrics.columns if column.startswith("osm_")]
for column in count_columns:
    zone_metrics[f"{column}_per_km2"] = zone_metrics[column] / zone_metrics["area_km2"]

zone_metrics["osm_attribute_richness"] = (zone_metrics[osm_count_columns] > 0).sum(axis=1) if osm_count_columns else 0
zone_metrics["osm_total_accessibility_elements"] = zone_metrics[osm_count_columns].sum(axis=1) if osm_count_columns else 0
zone_metrics["osm_total_accessibility_elements_per_km2"] = (
    zone_metrics["osm_total_accessibility_elements"] / zone_metrics["area_km2"]
)
zone_metrics["mapillary_available"] = zone_metrics.get("mapillary_images", 0) > 0
zone_metrics["attributes_present"] = zone_metrics[osm_count_columns].gt(0).apply(lambda row: ", ".join([col.replace("osm_", "") for col, ok in row.items() if ok]), axis=1)
zone_metrics["attributes_absent"] = zone_metrics[osm_count_columns].eq(0).apply(lambda row: ", ".join([col.replace("osm_", "") for col, missing in row.items() if missing]), axis=1)

zone_metrics.head()


,zone_id,area_km2,geometry,osm_audible_signals,osm_crossings,osm_incline,osm_kerbs,osm_ramps_or_wheelchair,osm_sidewalks,osm_steps,osm_surface_on_pedestrian_ways,osm_tactile_paving,osm_traffic_signals,mapillary_images,osm_audible_signals_per_km2,osm_crossings_per_km2,osm_incline_per_km2,osm_kerbs_per_km2,osm_ramps_or_wheelchair_per_km2,osm_sidewalks_per_km2,osm_steps_per_km2,osm_surface_on_pedestrian_ways_per_km2,osm_tactile_paving_per_km2,osm_traffic_signals_per_km2,mapillary_images_per_km2,osm_attribute_richness,osm_total_accessibility_elements,osm_total_accessibility_elements_per_km2,mapillary_available,attributes_present,attributes_absent
0,z01_02,0.436085,"POLYGON ((-3.71455 40.40671, -3.71448 40.40022...",27,101,20,10,19,80,25,209,63,13,0,61.914495,231.606076,45.862589,22.931295,43.569460,183.450357,57.328237,479.264057,144.467156,29.810683,0.000000,10,567,1300.204405,False,"audible_signals, crossings, incline, kerbs, ra...",
1,z02_02,0.633323,"POLYGON ((-3.71464 40.41572, -3.71455 40.40671...",37,84,19,28,59,50,46,278,116,25,0,58.422030,132.633799,30.000502,44.211266,93.159454,78.948690,72.632794,438.954714,183.160960,39.474345,0.000000,10,742,1171.598554,False,"audible_signals, crossings, incline, kerbs, ra...",
2,z03_02,0.716237,"POLYGON ((-3.72653 40.42465, -3.71474 40.42473...",28,51,40,22,122,66,47,360,136,25,0,39.093196,71.205463,55.847422,30.716082,170.334638,92.148247,65.620721,502.626800,189.881236,34.904639,0.000000,10,897,1252.378444,False,"audible_signals, crossings, incline, kerbs, ra...",
3,z00_06,0.312302,"POLYGON ((-3.6791 40.39791, -3.67007 40.39797,...",8,82,25,16,18,27,22,152,51,23,74,25.616238,262.566441,80.050744,51.232476,57.636536,86.454804,70.444655,486.708525,163.303518,73.646685,236.950203,10,424,1357.660622,True,"audible_signals, crossings, incline, kerbs, ra...",
4,z00_05,0.939383,"POLYGON ((-3.69089 40.39784, -3.6791 40.39791,...",28,190,37,31,36,51,57,369,127,35,0,29.806783,202.260316,39.387535,33.000367,38.323007,54.290927,60.678095,392.810825,135.195054,37.258479,0.000000,10,961,1023.011389,False,"audible_signals, crossings, incline, kerbs, ra...",


## 6. Puntuación para seleccionar zona piloto

La puntuacion no mide accesibilidad real. Mide idoneidad para el MVP: zonas con suficientes datos OSM variados y cobertura visual Mapillary para inspeccion/evaluacion futura.

Criterios:

- Riqueza de atributos OSM.
- Densidad total de elementos OSM relacionados con accesibilidad.
- Cobertura Mapillary si hay token.
- Penalizacion suave por ausencia de atributos criticos.


In [35]:
CRITICAL_ATTRIBUTES = ["osm_crossings", "osm_traffic_signals", "osm_tactile_paving", "osm_audible_signals", "osm_kerbs"]


def min_max(series: pd.Series) -> pd.Series:
    """Normalize a numeric series to 0..1, returning 0 when constant."""
    maximum = series.max()
    minimum = series.min()
    if maximum == minimum:
        return pd.Series(0.0, index=series.index)
    return (series - minimum) / (maximum - minimum)


zone_metrics["critical_attributes_present"] = sum(
    (zone_metrics[column] > 0).astype(int) for column in CRITICAL_ATTRIBUTES if column in zone_metrics.columns
)
zone_metrics["critical_attributes_missing"] = len([column for column in CRITICAL_ATTRIBUTES if column in zone_metrics.columns]) - zone_metrics["critical_attributes_present"]

zone_metrics["pilot_score"] = (
    0.40 * min_max(zone_metrics["osm_attribute_richness"])
    + 0.30 * min_max(zone_metrics["osm_total_accessibility_elements_per_km2"])
    + 0.20 * min_max(zone_metrics.get("mapillary_images_per_km2", pd.Series(0, index=zone_metrics.index)))
    + 0.10 * min_max(zone_metrics["critical_attributes_present"])
)

ranked_zones = zone_metrics.sort_values("pilot_score", ascending=False).copy()
ranked_zones[[
    "zone_id",
    "pilot_score",
    "osm_attribute_richness",
    "osm_total_accessibility_elements",
    "mapillary_images",
    "critical_attributes_present",
    "attributes_present",
    "attributes_absent",
]].head(20)


,zone_id,pilot_score,osm_attribute_richness,osm_total_accessibility_elements,mapillary_images,critical_attributes_present,attributes_present,attributes_absent
66,z10_03,0.808210,10,557,85,5,"audible_signals, crossings, incline, kerbs, ra...",
67,z10_05,0.731290,10,1129,99,5,"audible_signals, crossings, incline, kerbs, ra...",
44,z05_04,0.698236,10,1020,275,5,"audible_signals, crossings, incline, kerbs, ra...",
21,z04_02,0.695112,10,1339,243,5,"audible_signals, crossings, incline, kerbs, ra...",
3,z00_06,0.692624,10,424,74,5,"audible_signals, crossings, incline, kerbs, ra...",
63,z09_05,0.689927,10,1746,200,5,"audible_signals, crossings, incline, kerbs, ra...",
24,z05_01,0.682402,10,649,264,5,"audible_signals, crossings, incline, kerbs, ra...",
58,z07_04,0.680489,10,1987,163,5,"audible_signals, crossings, incline, kerbs, ra...",
59,z07_05,0.677980,10,745,261,5,"audible_signals, crossings, incline, kerbs, ra...",
33,z04_05,0.677149,10,1078,232,5,"audible_signals, crossings, incline, kerbs, ra...",


In [36]:
metrics_csv = DATA_PROCESSED / "madrid_accessibility_density_by_zone.csv"
geojson_path = DATA_PROCESSED / "madrid_accessibility_density_by_zone.geojson"

zone_metrics.drop(columns="geometry").to_csv(metrics_csv, index=False)
zone_metrics.to_file(geojson_path, driver="GeoJSON")

print(f"Saved metrics to {metrics_csv}")
print(f"Saved GeoJSON to {geojson_path}")


Saved metrics to /Users/luciacorsan/Desktop/tfm/tfm/data/processed/madrid_accessibility_density_by_zone.csv
Saved GeoJSON to /Users/luciacorsan/Desktop/tfm/tfm/data/processed/madrid_accessibility_density_by_zone.geojson


## 7. Mapas de densidad

In [ ]:
PILOT_LAYER_NAME = "Idoneidad para elegir la zona piloto"
OSM_LAYER_NAME = "Cobertura de datos de accesibilidad (OSM)"
MAPILLARY_LAYER_NAME = "Cobertura fotográfica de calles (Mapillary)"


def add_metric_layer(
    data: gpd.GeoDataFrame,
    column: str,
    layer_name: str,
    colors: list[str],
    show: bool = False,
) -> tuple[folium.FeatureGroup, LinearColormap]:
    """Create one thematic layer with its tooltip integrated.

    Args:
        data: Analysis zones and display-ready metric fields.
        column: Numeric column used to colour the zones.
        layer_name: Descriptive label shown in the map control.
        colors: Sequential colour palette from low to high values.
        show: Whether the layer is visible when the map opens.

    Returns:
        Feature group and colour scale used by the thematic layer.
    """
    values = pd.to_numeric(data[column], errors="coerce").fillna(0)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        maximum = minimum + 1.0
    colour_scale = LinearColormap(colors=colors, vmin=minimum, vmax=maximum)
    layer = folium.FeatureGroup(name=layer_name, show=show, overlay=True)

    def style_zone(feature: dict) -> dict:
        value = feature["properties"].get(column, 0)
        return {
            "color": "#334155",
            "weight": 0.75,
            "opacity": 0.70,
            "fillColor": colour_scale(value),
            "fillOpacity": 0.82,
        }

    tooltip = GeoJsonTooltip(
        fields=[
            "zone_id", "pilot_score_display", "osm_density_display",
            "mapillary_density_display", "osm_attribute_richness",
            "critical_attributes_present", "area_km2_display",
            "attributes_absent_display",
        ],
        aliases=[
            "Celda de análisis", "Idoneidad (0-1)", "Elementos OSM por km2",
            "Imágenes Mapillary por km2", "Tipos de atributos OSM presentes (máximo 10)",
            "Atributos críticos presentes", "Superficie efectiva (km2)",
            "Atributos sin datos",
        ],
        sticky=False,
        localize=True,
        labels=True,
        style=(
            "background-color: rgba(255,255,255,.97); color: #0f172a; "
            "font-family: Inter,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; "
            "font-size: 13px; line-height: 1.45; padding: 10px 12px; "
            "border: 1px solid #cbd5e1; border-radius: 10px; "
            "box-shadow: 0 8px 24px rgba(15,23,42,.18);"
        ),
    )
    folium.GeoJson(
        data.to_json(),
        style_function=style_zone,
        highlight_function=lambda _: {"color": "#0f172a", "weight": 2.4, "fillOpacity": 0.94},
        tooltip=tooltip,
        smooth_factor=0.5,
    ).add_to(layer)
    return layer, colour_scale


map_data = zone_metrics.copy()
if "mapillary_images_per_km2" not in map_data.columns:
    map_data["mapillary_images_per_km2"] = 0.0
if "critical_attributes_present" not in map_data.columns:
    map_data["critical_attributes_present"] = 0
map_data["pilot_score_display"] = map_data["pilot_score"].round(3)
map_data["osm_density_display"] = map_data["osm_total_accessibility_elements_per_km2"].round(1)
map_data["mapillary_density_display"] = map_data["mapillary_images_per_km2"].round(1)
map_data["area_km2_display"] = map_data["area_km2"].round(2)
map_data["attributes_absent_display"] = map_data["attributes_absent"].replace("", "Ninguno")

density_map = folium.Map(location=MADRID_CENTER, zoom_start=12, tiles=None, control_scale=True, prefer_canvas=True)
folium.TileLayer(tiles="CartoDB positron", name="Cartografia base", control=False, opacity=0.92).add_to(density_map)

pilot_layer, pilot_scale = add_metric_layer(
    map_data, "pilot_score", PILOT_LAYER_NAME,
    ["#faf5ff", "#ede9fe", "#ddd6fe", "#c4b5fd", "#8b5cf6"], show=True,
)
osm_layer, osm_scale = add_metric_layer(
    map_data, "osm_total_accessibility_elements_per_km2", OSM_LAYER_NAME,
    ["#f8fafc", "#dbeafe", "#60a5fa", "#2563eb", "#1e3a8a"],
)
mapillary_layer, mapillary_scale = add_metric_layer(
    map_data, "mapillary_images_per_km2", MAPILLARY_LAYER_NAME,
    ["#f8fafc", "#dcfce7", "#4ade80", "#16a34a", "#14532d"],
)
for layer in (pilot_layer, osm_layer, mapillary_layer):
    layer.add_to(density_map)

candidate_points = ranked_zones.head(5).to_crs(MADRID_CRS).copy()
candidate_points["geometry"] = candidate_points.geometry.centroid
candidate_points = candidate_points.to_crs(WGS84)
for rank, (_, candidate) in enumerate(candidate_points.iterrows(), start=1):
    folium.Marker(
        location=[candidate.geometry.y, candidate.geometry.x],
        tooltip=f"Candidata {rank}: {candidate['zone_id']} - idoneidad {candidate['pilot_score']:.3f}",
        icon=folium.DivIcon(
            icon_size=(30, 30), icon_anchor=(15, 15),
            html=f"<div class='candidate-marker'>{rank}</div>",
        ),
    ).add_to(pilot_layer)

GroupedLayerControl(
    groups={"Selecciona la información que quieres comparar": [pilot_layer, osm_layer, mapillary_layer]},
    exclusive_groups=True, collapsed=False, position="topright",
).add_to(density_map)

minimum_longitude, minimum_latitude, maximum_longitude, maximum_latitude = m30_boundary.total_bounds
density_map.fit_bounds(
    [[minimum_latitude, minimum_longitude], [maximum_latitude, maximum_longitude]], padding=(28, 28)
)

legend_config = {
    PILOT_LAYER_NAME: {
        "title": "Idoneidad para la zona piloto",
        "description": "Síntesis ponderada de riqueza OSM, densidad, Mapillary y atributos críticos.",
        "gradient": "linear-gradient(90deg,#faf5ff,#ede9fe,#ddd6fe,#c4b5fd,#8b5cf6)",
        "minimum": f"{map_data['pilot_score'].min():.2f}",
        "maximum": f"{map_data['pilot_score'].max():.2f}",
    },
    OSM_LAYER_NAME: {
        "title": "Cobertura de accesibilidad en OSM",
        "description": "Número de elementos con atributos de accesibilidad por km2.",
        "gradient": "linear-gradient(90deg,#f8fafc,#dbeafe,#60a5fa,#2563eb,#1e3a8a)",
        "minimum": f"{map_data['osm_total_accessibility_elements_per_km2'].min():,.0f}",
        "maximum": f"{map_data['osm_total_accessibility_elements_per_km2'].max():,.0f}",
    },
    MAPILLARY_LAYER_NAME: {
        "title": "Cobertura fotográfica de calles",
        "description": "Número de imágenes Mapillary disponibles por km2.",
        "gradient": "linear-gradient(90deg,#f8fafc,#dcfce7,#4ade80,#16a34a,#14532d)",
        "minimum": f"{map_data['mapillary_images_per_km2'].min():,.0f}",
        "maximum": f"{map_data['mapillary_images_per_km2'].max():,.0f}",
    },
}
map_name = density_map.get_name()

map_decorations = """
<style>
  .map-title-card{position:fixed;top:18px;left:52px;z-index:9999;width:min(430px,calc(100vw - 110px));background:rgba(255,255,255,.95);border:1px solid rgba(148,163,184,.55);border-radius:14px;box-shadow:0 12px 32px rgba(15,23,42,.16);padding:14px 17px;font-family:Inter,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;color:#0f172a;backdrop-filter:blur(8px)}
  .map-title-card .eyebrow{color:#0369a1;font-size:10px;font-weight:800;letter-spacing:.12em;text-transform:uppercase;margin-bottom:4px}.map-title-card h1{font-size:19px;line-height:1.2;margin:0;font-weight:750}.map-title-card p{color:#475569;font-size:12px;line-height:1.4;margin:5px 0 0}
  .map-legend-card{position:fixed;left:52px;bottom:28px;z-index:9999;width:320px;box-sizing:border-box;background:rgba(255,255,255,.96);border:1px solid rgba(148,163,184,.55);border-radius:14px;box-shadow:0 12px 32px rgba(15,23,42,.16);padding:13px 15px;font-family:Inter,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;color:#0f172a;backdrop-filter:blur(8px)}
  .map-legend-card h2{font-size:13px;line-height:1.25;margin:0 0 4px}.map-legend-card p{color:#475569;font-size:11px;line-height:1.35;margin:0 0 9px}.legend-gradient{width:100%;height:10px;border-radius:999px;border:1px solid rgba(100,116,139,.25)}.legend-limits{display:flex;justify-content:space-between;color:#334155;font-size:10px;font-weight:700;margin-top:3px}.legend-warning{border-top:1px solid #e2e8f0;color:#64748b;font-size:9.5px;line-height:1.35;margin-top:9px;padding-top:8px}
  .leaflet-control-layers{max-width:360px;border:1px solid rgba(148,163,184,.55)!important;border-radius:14px!important;box-shadow:0 12px 32px rgba(15,23,42,.16)!important;padding:10px 12px!important;font-family:Inter,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif}.leaflet-control-layers-group-name{display:block;color:#334155;font-size:10px;font-weight:800;letter-spacing:.06em;line-height:1.35;text-transform:uppercase;margin:2px 0 8px}.leaflet-control-layers-group label{display:flex;align-items:center;margin:6px 0;padding:8px 10px;background:#f8fafc;border:1px solid #e2e8f0;border-radius:9px;cursor:pointer;transition:background .15s,border-color .15s,transform .15s}.leaflet-control-layers-group label:hover{background:#f1f5f9;border-color:#94a3b8;transform:translateY(-1px)}.leaflet-control-layers-group label:has(input:checked){background:#e0f2fe;border-color:#38bdf8}.leaflet-control-layers-group label span{color:#0f172a;font-size:12px;line-height:1.25}.leaflet-control-layers-selector{margin-right:8px;accent-color:#0284c7;transform:scale(1.12)}
  .candidate-marker{width:28px;height:28px;display:flex;align-items:center;justify-content:center;color:#fff;background:#7c3aed;border:2px solid #fff;border-radius:50%;box-shadow:0 3px 10px rgba(15,23,42,.30);font:800 13px/1 Inter,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif}
  @media(max-width:720px){.map-title-card{left:44px;width:calc(100vw - 60px)}.map-legend-card{left:12px;bottom:24px;width:calc(100vw - 24px)}.leaflet-control-layers{max-width:280px}}
</style>
<div class="map-title-card"><div class="eyebrow">TFM · Madrid · Interior de la M-30</div><h1>Cobertura de datos para seleccionar la zona piloto</h1><p>Selecciona una vista y pasa el cursor por una celda para consultar sus indicadores.</p></div>
<div id="metric-legend" class="map-legend-card"><h2 id="legend-title"></h2><p id="legend-description"></p><div id="legend-gradient" class="legend-gradient"></div><div class="legend-limits"><span id="legend-minimum"></span><span id="legend-maximum"></span></div><div class="legend-warning">Valores más altos = mayor disponibilidad relativa de datos. La cobertura de datos no demuestra por sí sola que una zona o ruta sea accesible.</div></div>
"""
density_map.get_root().html.add_child(folium.Element(map_decorations))

legend_script = f"""
document.addEventListener('DOMContentLoaded', function() {{
const metricLegends = {json.dumps(legend_config, ensure_ascii=False)};
function updateMetricLegend(layerName) {{
  const config = metricLegends[layerName]; if (!config) return;
  document.getElementById('legend-title').textContent = config.title;
  document.getElementById('legend-description').textContent = config.description;
  document.getElementById('legend-gradient').style.background = config.gradient;
  document.getElementById('legend-minimum').textContent = config.minimum + ' - menor';
  document.getElementById('legend-maximum').textContent = config.maximum + ' - mayor';
}}
updateMetricLegend({json.dumps(PILOT_LAYER_NAME)});
{map_name}.on('overlayadd', function(event) {{ updateMetricLegend(event.name); }});
}});
"""
density_map.get_root().script.add_child(folium.Element(legend_script))

density_map_path = MAPS_DIR / "madrid_accessibility_density_map.html"
memory_map_path = MAPS_DIR / "madrid_accessibility_density_map_memoria.html"
density_map.save(density_map_path)
density_map.save(memory_map_path)
density_map


In [ ]:
def make_colour_map(name: str, colours: list[str]) -> LinearSegmentedColormap:
    """Create a continuous colour map from a thesis-ready palette.

    Args:
        name: Internal Matplotlib colour-map name.
        colours: Ordered colours from low to high values.

    Returns:
        Continuous Matplotlib colour map.
    """
    return LinearSegmentedColormap.from_list(name, colours)


static_palettes = {
    "pilot_score": make_colour_map(
        "pilot_violet", ["#faf5ff", "#ede9fe", "#ddd6fe", "#c4b5fd", "#8b5cf6"]
    ),
    "osm_attribute_richness": make_colour_map(
        "osm_richness_blue", ["#f8fafc", "#e0f2fe", "#bae6fd", "#38bdf8", "#0369a1"]
    ),
    "osm_total_accessibility_elements_per_km2": make_colour_map(
        "osm_density_blue", ["#f8fafc", "#dbeafe", "#93c5fd", "#60a5fa", "#1e3a8a"]
    ),
    "mapillary_images_per_km2": make_colour_map(
        "mapillary_green", ["#f8fafc", "#dcfce7", "#bbf7d0", "#4ade80", "#14532d"]
    ),
}

static_panels = [
    ("pilot_score", "Idoneidad para seleccionar la zona piloto", "Índice compuesto (0-1)"),
    ("osm_attribute_richness", "Diversidad temática de datos OSM", "Tipos presentes (máximo 10)"),
    ("osm_total_accessibility_elements_per_km2", "Densidad de datos de accesibilidad OSM", "Elementos por km²"),
    ("mapillary_images_per_km2", "Cobertura fotográfica de calles", "Imágenes Mapillary por km²"),
]
static_data = zone_metrics.to_crs(MADRID_CRS).copy()
static_boundary = m30_boundary.to_crs(MADRID_CRS)
static_candidates = ranked_zones.head(5).to_crs(MADRID_CRS).copy()
static_candidates["geometry"] = static_candidates.geometry.centroid

figure_background = "#f8fafc"
fig, axes = plt.subplots(2, 2, figsize=(14, 12), facecolor=figure_background)
axes = axes.flatten()
minimum_x, minimum_y, maximum_x, maximum_y = static_data.total_bounds
padding_metres = 350

for axis, (column, title, legend_label) in zip(axes, static_panels):
    axis.set_facecolor("#ffffff")
    static_data.plot(
        column=column,
        ax=axis,
        cmap=static_palettes[column],
        legend=True,
        edgecolor="#64748b",
        linewidth=0.45,
        legend_kwds={"label": legend_label, "shrink": 0.72, "pad": 0.02},
    )
    static_boundary.boundary.plot(ax=axis, color="#0f172a", linewidth=1.15)
    axis.set_xlim(minimum_x - padding_metres, maximum_x + padding_metres)
    axis.set_ylim(minimum_y - padding_metres, maximum_y + padding_metres)
    axis.set_title(title, loc="left", fontsize=14, fontweight="bold", color="#0f172a", pad=10)
    axis.set_axis_off()

for rank, (_, candidate) in enumerate(static_candidates.iterrows(), start=1):
    axes[0].scatter(
        candidate.geometry.x, candidate.geometry.y, s=145, facecolor="#ffffff",
        edgecolor="#7c3aed", linewidth=2.2, zorder=5,
    )
    axes[0].text(
        candidate.geometry.x, candidate.geometry.y, str(rank), ha="center", va="center",
        color="#6d28d9", fontsize=9.5, fontweight="bold", zorder=6,
    )

axes[0].text(
    0.02, 0.025, "1-5 · zonas candidatas mejor clasificadas", transform=axes[0].transAxes,
    fontsize=9.5, color="#4c1d95",
    bbox={"boxstyle": "round,pad=0.5", "facecolor": "#f5f3ff", "edgecolor": "#c4b5fd", "alpha": 0.96},
)

fig.suptitle(
    "Cobertura de datos de accesibilidad en el interior de la M-30",
    x=0.06, y=0.985, ha="left", fontsize=22, fontweight="bold", color="#0f172a",
)
fig.text(
    0.06, 0.952,
    "Comparación espacial de los indicadores empleados para seleccionar el área piloto",
    ha="left", fontsize=12, color="#475569",
)
fig.text(
    0.06, 0.018,
    "Nota: una mayor cobertura de datos reduce incertidumbre, pero no demuestra por sí sola que una zona sea accesible.",
    ha="left", fontsize=9.5, color="#64748b",
)
fig.text(
    0.94, 0.018, "Fuente: OpenStreetMap y Mapillary · Elaboración propia",
    ha="right", fontsize=9.5, color="#64748b",
)
fig.subplots_adjust(left=0.045, right=0.965, top=0.91, bottom=0.06, wspace=0.08, hspace=0.15)

static_map_path = MAPS_DIR / "madrid_accessibility_density_static.png"
fig.savefig(static_map_path, dpi=300, bbox_inches="tight", facecolor=figure_background)
plt.close(fig)
static_map_path
